ベース：白地図 CARTO Light No Labels

国境線：HDX OCHA "Syrian Arab Republic – Subnational Administrative Boundaries"
https://data.humdata.org/dataset/cod-ab-syr
syr_admin0.geojson

地域線：HDX OCHA "Syrian Arab Republic – Subnational Administrative Boundaries"
https://data.humdata.org/dataset/cod-ab-syr
syr_admin1.geojson

分析手法：GeoPandas Dissolve

対象地域：
・Syria (National Scale)

その他：
・シリア14州（Governorates）へ独自セクター名を付与
・Northwest / Northeast / Central / South の4つへ分類
・GeoPandas dissolve により、Governorate境界を統合
・Humanitarian Sector Map として再構築

In [1]:
# Core Libraries
# 地図作成とベクタ処理に必要なライブラリを読み込む

import geopandas as gpd

import folium

/Users/marisa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ETL Extract
# シリア14州の行政界データを読み込む
# 列名とCRSを確認する

admin1 = gpd.read_file("syr_admin1.geojson")

print(admin1.columns)
print(admin1.crs)

Index(['adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode',
       'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode',
       'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang', 'lang1',
       'lang2', 'lang3', 'adm1_ref_name', 'center_lat', 'center_lon',
       'geometry'],
      dtype='object')
EPSG:4326


In [3]:
# Check
# Governorate名の表記を確認する

print(admin1["adm1_name"].unique())

['Al-Hasakeh' 'Aleppo' 'Ar-Raqqa' 'As-Sweida' 'Damascus' "Dar'a"
 'Deir-ez-Zor' 'Hama' 'Homs' 'Idleb' 'Lattakia' 'Quneitra'
 'Rural Damascus' 'Tartous']


In [4]:
# ETL Transform
# Governorateごとに独自セクター名を付与するための分類表を作成

sector_map = {

    "Aleppo": "Northwest",
    "Idleb": "Northwest",
    "Lattakia": "Northwest",

    "Al-Hasakeh": "Northeast",
    "Ar-Raqqa": "Northeast",
    "Deir-ez-Zor": "Northeast",

    "Damascus": "South",
    "Rural Damascus": "South",
    "Dar'a": "South",
    "Quneitra": "South",
    "As-Sweida": "South",

    "Homs": "Central",
    "Hama": "Central",
    "Tartous": "Central"
}

In [5]:
# ETL Transform
# 14州へ人道支援セクター名を付与する

admin1["sector"] = admin1["adm1_name"].map(sector_map)

print(admin1[["adm1_name", "sector"]])

         adm1_name     sector
0       Al-Hasakeh  Northeast
1           Aleppo  Northwest
2         Ar-Raqqa  Northeast
3        As-Sweida      South
4         Damascus      South
5            Dar'a      South
6      Deir-ez-Zor  Northeast
7             Hama    Central
8             Homs    Central
9            Idleb  Northwest
10        Lattakia  Northwest
11        Quneitra      South
12  Rural Damascus      South
13         Tartous    Central


In [6]:
# ETL Transform
# セクターごとにGovernorate境界を統合する

sector_gdf = admin1.dissolve(by="sector")

print(sector_gdf)

                                                    geometry   adm1_name  \
sector                                                                     
Central    MULTIPOLYGON (((39.14117 35.3266, 39.14281 35....        Hama   
Northeast  POLYGON ((41.2826 35.48615, 41.28178 35.47528,...  Al-Hasakeh   
Northwest  POLYGON ((36.23237 35.74159, 36.23464 35.73857...      Aleppo   
South      POLYGON ((37.09276 32.46435, 37.09227 32.46408...   As-Sweida   

          adm1_name1 adm1_name2 adm1_name3 adm1_pcode             adm0_name  \
sector                                                                        
Central         حماة       None       None       SY05  Syrian Arab Republic   
Northeast     الحسكة       None       None       SY08  Syrian Arab Republic   
Northwest        حلب       None       None       SY02  Syrian Arab Republic   
South       السويداء       None       None       SY13  Syrian Arab Republic   

                          adm0_name1 adm0_name2 adm0_name3  ... vali

In [7]:
# Check
# 4つのセクターへ正しく統合されたか確認する

print(sector_gdf.index)

# Check
# dissolve後もCRSが保持されているか確認する

print(sector_gdf.crs)

Index(['Central', 'Northeast', 'Northwest', 'South'], dtype='object', name='sector')
EPSG:4326


In [8]:
# ETL Transform
# Folium表示に必要な列だけ残す

sector_gdf = sector_gdf.reset_index()
sector_gdf = sector_gdf[["sector", "geometry"]]

In [9]:
# Map Setup
# 白地図ベースを作成

m = folium.Map(
    location=[34.8, 38.5], 
    zoom_start=7,
    tiles='https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
)

In [10]:
# ETL Load
# Humanitarian Sector Mapを地図へ追加

folium.GeoJson(
    sector_gdf,
    name="Humanitarian Sectors",
    style_function=lambda x: {
        "fillColor": "#f4a261",
        "color": "#e76f51",
        "weight": 2,
        "fillOpacity": 0.35
    }
).add_to(m)

In [11]:
# ETL Load
# 国境線を地図へ追加

folium.GeoJson('syr_admin0.geojson', name='Country',
               style_function=lambda x: {'color': 'black', 'weight': 3, 'fillOpacity': 0}).add_to(m)

In [12]:
# ETL Load
# 隣国ラベルを地図へ追加

neighbors = {
    "TÜRKIYE": [37.5, 37.5], 
    "IRAQ": [34.5, 42.0],         
    "JORDAN": [31.9, 36.5],       
    "LEBANON": [34.2, 35.0]       
}

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="font-size: 14pt; font-weight: bold; color: gray; white-space: nowrap; text-align: center; width: 100px; margin-left: -50px;">{name}</div>'''
        )
    ).add_to(m)

In [13]:
# ETL Load
# 4つの人道支援セクター名を地図へ追加

sectors = {
    "Northwest": [36.1, 37.3],
    "Northeast": [36.1, 40.2],
    "Central": [34.8, 38.0],
    "South": [33.3, 36.9]
}

for name, coords in sectors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="
                font-size: 14pt;
                color: #5c4033;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 140px;
                margin-left: -70px;
                ">
                {name}
                </div>'''
        )
    ).add_to(m)

In [14]:
# ETL Load
# HTMLマップとして保存

m.save('12_syria_vector_dissolve.html')